In [1]:
import numpy as np
from scipy.stats import rankdata, norm
from statsmodels.distributions.copula.api import GaussianCopula

In [2]:
data = {
    1: {
        "agv": np.array([2.24, 2.96, 2.45]),
        "op":  np.array([10.33, 9.36, 9.18])
    },
    2: {
        "agv": np.array([5.32, 4.21, 4.68]),
        "op":  np.array([19.15, 16.04, 19.48])
    },
    3: {
        "agv": np.array([7.37, 7.52, 8.28]),
        "op":  np.array([21.10, 26.31, 28.39])
    },
    4: {
        "agv": np.array([10.98, 10.51, 10.53]),
        "op":  np.array([34.25, 36.48, 25.71])
    }
}

In [3]:
from scipy.stats import rankdata, norm
from statsmodels.distributions.copula.api import GaussianCopula

# maak 1 dataset van alle observaties
agv_all = np.concatenate([data[i]["agv"] for i in data])
op_all  = np.concatenate([data[i]["op"] for i in data])

# rank transform → uniform [0,1]
u_agv = rankdata(agv_all) / (len(agv_all) + 1)
u_op  = rankdata(op_all) / (len(op_all) + 1)

data_uniform = np.column_stack([u_agv, u_op])

# fit copula
copula = GaussianCopula(k_dim=2)
copula.fit_corr_param(data_uniform)

np.float64(0.9450008187146683)

In [4]:
#extrapoleren naar 5,6,7 pakjes
items = np.array([1,2,3,4])

agv_means = np.array([np.mean(data[i]["agv"]) for i in items])
op_means  = np.array([np.mean(data[i]["op"]) for i in items])

# lineaire fit
agv_coef = np.polyfit(items, agv_means, 1)
op_coef  = np.polyfit(items, op_means, 1)

def extrapolate_mean(n, coef):
    return coef[0]*n + coef[1]

In [5]:
def get_stats(n_items):
    agv = data[n_items]["agv"]
    op = data[n_items]["op"]
    
    return {
        "agv_mu": np.mean(agv),
        "agv_sigma": np.std(agv, ddof=1),
        "op_mu": np.mean(op),
        "op_sigma": np.std(op, ddof=1)
    }

In [6]:
def sample_service_time(n_items):
    
    # 1. sample uit copula
    samples = copula.rvs(1)
    samples = np.atleast_2d(samples)
    u1, u2 = samples[0]
    
    # 2. bepaal mean + std 
    if n_items in data:
        stats = get_stats(n_items)
        agv_mu = stats["agv_mu"]
        agv_sigma = stats["agv_sigma"]
        op_mu = stats["op_mu"]
        op_sigma = stats["op_sigma"]
        
    else:
        agv_mu = extrapolate_mean(n_items, agv_coef)
        op_mu  = extrapolate_mean(n_items, op_coef)
        
        agv_sigma = 0.1 * agv_mu
        op_sigma  = 0.1 * op_mu
    
    # 3. transformeer copula → echte tijden
    agv_time = norm.ppf(u1, loc=agv_mu, scale=agv_sigma)
    op_time  = norm.ppf(u2, loc=op_mu, scale=op_sigma)
    
    # 4. ondergrens (kan uitgezet worden)
    agv_time = max(agv_time, 1.0)
    op_time  = max(op_time, 5.0)
    
    return agv_time, op_time

In [7]:
for n in [1,2,3,4,5,6,7,8]:
    agv, op = sample_service_time(n)
    print(f"{n} items -> AGV: {agv:.2f}s, Operator: {op:.2f}s")

1 items -> AGV: 2.79s, Operator: 9.26s
2 items -> AGV: 5.32s, Operator: 18.40s
3 items -> AGV: 8.53s, Operator: 28.35s
4 items -> AGV: 10.30s, Operator: 30.57s
5 items -> AGV: 12.60s, Operator: 38.37s
6 items -> AGV: 14.78s, Operator: 46.56s
7 items -> AGV: 19.94s, Operator: 54.28s
8 items -> AGV: 24.09s, Operator: 66.39s


In [8]:
def simulate_queue(n_orders, arrival_rate):
    
    arrivals = np.cumsum(np.random.exponential(1/arrival_rate, n_orders))
    
    operator_free_time = 0
    
    waiting_times = []
    agv_departure_times = []
    
    for i in range(n_orders):
        
        arrival = arrivals[i]
        
        # aantal items
        n_items = np.random.randint(1,5)
        
        # copula sample
        agv_time, op_time = sample_service_time(n_items)
        
        # wanneer kan proces starten?
        start_time = max(arrival, operator_free_time)
        
        # operator bepaalt queue!
        operator_finish = start_time + op_time
        
        # AGV vertrekt eerder
        agv_departure = start_time + agv_time
        
        # wachttijd = wachten op operator
        waiting = start_time - arrival
        
        # update
        operator_free_time = operator_finish
        
        # opslaan
        waiting_times.append(waiting)
        agv_departure_times.append(agv_departure)
    
    return {
        "avg_waiting_time": np.mean(waiting_times),
        "avg_agv_time_in_system": np.mean(np.array(agv_departure_times) - arrivals),
        "utilization_operator": operator_free_time / arrivals[-1]
    }

In [12]:
#arrival rate en hoeveelheid orders wordt bepaald door de agv (zelf aanpassen)

result = simulate_queue(n_orders=1000, arrival_rate=1)

print("Average waiting time:", result["avg_waiting_time"])
print("avg agv time in system:", result["avg_agv_time_in_system"])
print("utilization operator:", result["utilization_operator"])

Average waiting time: 10059.326865778952
avg agv time in system: 10065.647987478338
utilization operator: 20.847974959316176
